# Milestone 5 lab, notebook 2 of 7: Tasks and arrows

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| **2. Tasks and arrows** (you are here) | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

In notebook 1 the rule was "the transform needs the load to finish first". Airflow lets you write that rule down. This notebook reads the file where it is written, and runs it.

In [1]:
from lab_helpers import at, MILESTONE_DIR, report_is_right, schedule_warehouse, show_doc, show_runs, start_fresh, table

at.wait_until_ready()
start_fresh()

Airflow has no runs. The warehouse is empty. The order system is back at its first morning.


`start_fresh` gives every notebook of this milestone the same clean start. `airflow_tools.py` is a small file in this milestone's folder. Each of its functions runs a normal `airflow` command inside the Airflow container, and it shows the command in its text.

## Step 1: The file

Airflow reads Python files from a folder. This is the one for Arden. Read it from top to bottom.

In [2]:
print((MILESTONE_DIR / "dags" / "daily_orders_dag.py").read_text())

"""The daily orders pipeline, on a schedule.

One DAG, four tasks, three arrows:

    set_source_clock  >>  extract_and_load  >>  transform_and_test  >>  publish_report

Airflow decides WHEN each task runs, in WHAT ORDER, and what happens when one FAILS.
The work itself is the same program as before: `python -m daily_orders <step>`.
"""

from datetime import datetime, timedelta

from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import DAG, CronDataIntervalTimetable

ARDEN = "America/Chicago"
PYTHON = "/opt/arden-venv/bin/python"  # the pipeline's own virtual environment, inside the image
PIPELINE = f"cd /opt/arden/src/m5_on_a_schedule && {PYTHON} -m daily_orders"

# Every run is FOR one business day: its logical date. In templates that day is {{ ds }}.
# The run HAPPENS the morning after, when that day is over. That is the same for a
# scheduled run, a backfill, and a run a person triggers by hand.
BUSINESS_DAY = "{{ ds }}"
RUN_DATE = "{{ macros.ds_add

It is short, and it does no work itself. Three things are in it.

**Tasks.** A **task** is one step of the job. There are four. Each one is a `BashOperator`: "run this command". The commands are the steps you ran by hand in notebook 1.

**Arrows.** The last line is the heart of the file:

```python
set_source_clock >> extract_and_load >> transform_and_test >> publish_report
```

`a >> b` means: **b needs a to finish first.** That is a **dependency**. It is exactly the rule that two cron lines could not express.

**The DAG.** Tasks and arrows together are a **DAG**: a directed acyclic graph. *Directed*: every arrow points one way. *Acyclic*: you can never follow the arrows back to where you started, so there is always a first task and a last one.

```
set_source_clock  ──▶  extract_and_load  ──▶  transform_and_test  ──▶  publish_report
```

One task is there only because of this project. `set_source_clock` moves the fake order system to the morning of the run. A real order system has no clock you can move, and a real DAG has no such task.

## Step 2: Airflow has read it

Ask Airflow which DAGs it knows, and which tasks this one has.

In [3]:
print(at.airflow("dags", "list", "--columns", "dag_id,is_paused"))

dag_id                    | is_paused
==========================+==========
daily_orders              | False    
orders_every_five_minutes | True     
                                     



In [4]:
print(at.airflow("tasks", "list", "daily_orders"))

extract_and_load
publish_report
set_source_clock
transform_and_test



You can see the same thing as a picture. Open http://localhost:8080 in a browser, click **daily_orders**, and choose the graph view. No password is needed.

## Step 3: Run it once

`airflow dags test` runs one whole run, here and now, and prints what each task says. Run it for the business day Sunday 7 June.

In [5]:
output = at.airflow("dags", "test", "daily_orders", "2026-06-07T12:00:00+00:00", with_logs=True)

for line in output.splitlines():
    if "Task started" in line:
        print("─" * 60)
    for marker in ["The order system's clock", "extract-load:", "transform-test:", "Arden daily orders", "Yesterday ", "Day before ", "Change  "]:
        if marker in line:
            print("   ", line.split("[info     ]")[1].split("[airflow")[0].strip())

────────────────────────────────────────────────────────────
    The order system's clock: 2026-06-08 07:00:00 (America/Chicago)
────────────────────────────────────────────────────────────
    extract-load: run 2026-06-08_1 landed 226 rows in 10 API calls.
────────────────────────────────────────────────────────────
    transform-test: dbt built 7 models and passed 31 tests on the tables.
────────────────────────────────────────────────────────────
    Arden daily orders. Report for the morning of Mon 2026-06-08
    Yesterday   Sun 2026-06-07    30 orders      $807.11
    Day before  Sat 2026-06-06    33 orders      $786.71
    Change                         -3 orders      +$20.40


Four tasks ran, one after the other, each only when the one before it had finished. The report at the end is the one you know: Sunday, 30 orders, $807.11.

Check the result against the answer key.

In [6]:
print("The report equals the answer key for the morning of 8 June:", report_is_right("2026-06-08"))

con = schedule_warehouse()
table(con, "SELECT run_id, rows_landed, api_calls, models_built, tests_passed, tests_failed FROM ops.load_runs")

The report equals the answer key for the morning of 8 June: True


,run_id,rows_landed,api_calls,models_built,tests_passed,tests_failed
0,2026-06-08_1,226,10,7,31,0


In [7]:
con.close()

## Step 4: What Airflow adds, and what it does not

Look at what did the work: the same Python program, the same dbt models, the same 31 table tests. Airflow added none of that.

| Airflow does | Airflow does not |
|---|---|
| Start the run at the right time | Read the order system |
| Run the tasks in the order of the arrows | Know what an order is |
| Wait for a task before it starts the next | Hold any business rule |
| Try a failed task again | Make a step safe to repeat. That was milestone 3 |
| Keep a history of every run, with logs | |

An orchestrator is a conductor. It plays no instrument.

In [8]:
show_doc("LLD.md", "3. The DAG (`dags/daily_orders_dag.py`)")

**From `docs/LLD.md`**

> ## 3. The DAG (`dags/daily_orders_dag.py`)
> 
> ```python
> with DAG(
>     dag_id="daily_orders",
>     schedule=CronDataIntervalTimetable("0 7 * * *", timezone="America/Chicago"),
>     start_date=datetime(2026, 6, 1, 12, 0),
>     end_date=datetime(2026, 6, 17, 12, 0),
>     catchup=False,
>     max_active_runs=1,
> ) as dag:
>     ...
>     set_source_clock >> extract_and_load >> transform_and_test >> publish_report
> ```
> 
> | Setting | Value | Meaning |
> |---|---|---|
> | `schedule` | `0 7 * * *` in `America/Chicago`, with data intervals | Every day at 07:00 in Arden. Each run covers the 24 hours before it |
> | `start_date`, `end_date` | 1 June to 17 June 2026 | The first and the last business day that gets a run |
> | `catchup` | `False` | Missed days are not run by themselves |
> | `max_active_runs` | 1 | One run at a time |
> 
> The cron line `0 7 * * *` reads: minute 0, hour 7, every day of the month, every month, every day of the week.
> 
> | Task | Command | Retries |
> |---|---|---|
> | `set_source_clock` | `python -m order_system.clock --morning {{ macros.ds_add(ds, 1) }}` | 0 |
> | `extract_and_load` | `python -m daily_orders extract-load --run-date {{ macros.ds_add(ds, 1) }}` | 3, 20 seconds apart |
> | `transform_and_test` | `python -m daily_orders transform-test` | 0 |
> | `publish_report` | `python -m daily_orders report` | 0 |
> 
> `{{ ds }}` is the logical date as text, like `2026-06-07`: the business day the run is for. `{{ macros.ds_add(ds, 1) }}` is the day after: the morning the run happens. Airflow fills them in for each run.
> 
> `--run-date` is a guard. If the order system's clock is not on that morning, the step stops before it loads anything.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Task** | "A task is one step of the job." | `extract_and_load` |
| **Dependency** | "A dependency is a rule that says task B needs task A to finish first." | `extract_and_load >> transform_and_test` |
| **DAG** | "DAG means directed acyclic graph." The tasks of a job and the arrows between them | `daily_orders`: four tasks, three arrows |

The definitions are from the D8Loop chapter [Orchestration & Scheduling](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/orchestration-and-scheduling).

## What you now know

- A task is one step. A dependency says one task needs another to finish first. A DAG is the tasks and the arrows.
- The DAG file holds no business logic. Each task starts a step of the same program as before.
- `airflow dags test` runs one whole run and shows it.

**Say it in an interview:** "My DAG has four tasks in a line: extract and load, then dbt build with its tests, then publish. The tasks are thin: each one runs a step of a program that also works without Airflow. Airflow gives me order, timing, retries and a run history. It holds no business logic."

Next: [notebook 3](03_which_day_is_this_run_for.ipynb) asks a question that sounds simple: which day is a run for?